# 01 — Run Experiments (pi0.5, LIBERO / LIBERO-PRO)

Thin driver notebook. All logic lives in the `pnp` package; results go to Supabase.
Run top to bottom on a Colab GPU runtime.

## 1. Secrets + install (clone private repo, editable)

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
![ -d pnp-vla ] || git clone -q https://$GH_PAT@github.com/ArjunS07/pnp-vla.git
!cd pnp-vla && git pull -q && pip install -q -e '.[sim]'

## 2. Environment + model

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()   # restart runtime + re-run if it upgrades torch

In [ ]:
from pnp import libero_env, models
from pnp.store import SupabaseStore
from pnp.rollout import run_controlled_slice

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()

## 3. (optional) Verify the noise/no-op contract before spending GPU time

In [ ]:
from pnp.pnp import assert_pnp_noop
from pnp.libero_env import build_final_episodes, make_env, obs_to_policy
from pnp.config import NUM_STEPS_WAIT, LIBERO_DUMMY_ACTION, CAMERAS

_ep = build_final_episodes()[0]
_env = make_env(_ep['bddl_path'])
_env.reset(); policy.reset(); _obs = _env.set_init_state(_ep['init_state'])
for _ in range(NUM_STEPS_WAIT): _obs, *_ = _env.step(LIBERO_DUMMY_ACTION)
_batch = preprocess(obs_to_policy(_obs, _ep['task_desc']))
# assert_pnp_noop expects the model's sample_actions positional args; see verify docs.
_env.close()
print('slice episode ready:', _ep['suite'], _ep['task_idx'])

## 4. Controlled 80-episode slice (4 methods, paired, RNG-isolated)

In [ ]:
episodes = build_final_episodes()   # 8 stock tasks x 10 episodes
n = run_controlled_slice(store, policy, preprocess, device, episodes,
                         experiment='slice-v1')
print(f'logged {n} rollouts to experiment=slice-v1  (~{store.bytes_written/1e6:.1f} MB blobs)')

## 5. LIBERO-PRO 600-episode stretch

Requires the LIBERO-PRO assets + `pnp.libero_pro` setup (see the LIBERO-PRO setup notebook).
Uncomment once `libero_pro.py` is populated.

In [ ]:
# from pnp import libero_pro
# from pnp.rollout import run_pro
# libero_pro.setup(...)
# pro_eps = libero_pro.build_libero_pro_episodes(...)
# run_pro(store, policy, preprocess, device, pro_eps, experiment='pro-v1')